# Plant Disease Classification Training

Use this notebook to train a plant disease image classifier from a folder-based dataset. Keep the dataset inside `dataset/`; outputs are saved into `model/` and `trained_model/`.

## Expected Dataset Format

Each disease or healthy class must have its own folder:

```text
dataset/
  Tomato___Early_blight/
    img001.jpg
  Tomato___healthy/
    img001.jpg
  Potato___Late_blight/
    img001.jpg
```

In [ ]:
from pathlib import Path
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()
DATASET_DIR = BASE_DIR / "dataset"
MODEL_DIR = BASE_DIR / "model"
TRAINED_MODEL_DIR = BASE_DIR / "trained_model"

MODEL_DIR.mkdir(exist_ok=True)
TRAINED_MODEL_DIR.mkdir(exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
EPOCHS = 15

print("TensorFlow:", tf.__version__)
print("Dataset folder:", DATASET_DIR)
print("Model metadata folder:", MODEL_DIR)
print("Trained model folder:", TRAINED_MODEL_DIR)

In [ ]:
if not DATASET_DIR.exists():
    raise FileNotFoundError(f"Dataset folder not found: {DATASET_DIR}")

class_dirs = sorted([p for p in DATASET_DIR.iterdir() if p.is_dir()])
if len(class_dirs) < 2:
    raise ValueError("Add at least two class folders inside dataset/ before training.")

print("Classes found:")
for class_dir in class_dirs:
    image_count = sum(1 for p in class_dir.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"})
    print(f"- {class_dir.name}: {image_count} images")

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
num_classes = len(class_names)

label_map = {i: name for i, name in enumerate(class_names)}
with open(MODEL_DIR / "label_map.json", "w", encoding="utf-8") as f:
    json.dump(label_map, f, indent=2)

print("Number of classes:", num_classes)
print(label_map)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="data_augmentation")

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs, name="plant_disease_classifier")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()
with open(MODEL_DIR / "model_architecture.json", "w", encoding="utf-8") as f:
    f.write(model.to_json())

In [ ]:
checkpoint_path = TRAINED_MODEL_DIR / "best_plant_disease_model.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks,
)

In [ ]:
final_model_path = TRAINED_MODEL_DIR / "plant_disease_model.keras"
model.save(final_model_path)

training_summary = {
    "image_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "classes": class_names,
    "final_model": str(final_model_path),
    "best_model": str(checkpoint_path),
}

with open(MODEL_DIR / "training_summary.json", "w", encoding="utf-8") as f:
    json.dump(training_summary, f, indent=2)

print("Saved final model to:", final_model_path)
print("Saved best model to:", checkpoint_path)

In [ ]:
def predict_image(image_path):
    image_path = Path(image_path)
    img = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)
    arr = np.expand_dims(arr, axis=0)
    preds = model.predict(arr)[0]
    top_index = int(np.argmax(preds))
    return {
        "class_index": top_index,
        "class_name": class_names[top_index],
        "confidence": round(float(preds[top_index]) * 100, 2),
    }

# Example after training:
# predict_image(DATASET_DIR / "Tomato___healthy" / "example.jpg")